# `torch_scatter` vs `dpvo.onnx_mods` (SoftAgg pattern)

This notebook checks that `onnx_mods.torch_scatter_softmax` and `onnx_mods.torch_scatter_sum` match `torch_scatter` for the same usage as `blocks.py` (SoftAgg / SoftAggBasic):

```python
w = onnx_mods.torch_scatter_softmax(self.g(x), jx, dim=1)
y = onnx_mods.torch_scatter_sum(self.f(x) * w, jx, dim=1)
```

**Environment:** use the project conda env, e.g. `conda activate dpvo`, or from a shell: `conda run -n dpvo jupyter notebook`.

During `torch.onnx.export`, you may see `TracerWarning` about `dim_size` from `index.max().item()`. That value is baked in as a constant for the traced graph (typical when `jx` is fixed-length padded edges). For a different number of groups at runtime, pass an explicit `dim_size` into the helpers so the graph does not rely on Python scalars from indices.

In [1]:
import io
import torch
import torch.nn as nn
import torch_scatter
import sys
sys.path.append('/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/dpvo/')
import onnx_mods as om

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

## 1. Same pattern as `SoftAgg` (`blocks.py`)

`jx` comes from `torch.unique(ix, return_inverse=True)[1]`, shape `[E]`.

In [2]:
def run_softagg_pattern(B, E, D, dim=1, basic_g=False):
    ix = torch.randint(0, max(3, E // 3), (E,), device=device)
    _, jx = torch.unique(ix, return_inverse=True)

    x = torch.randn(B, E, D, device=device)
    f = nn.Linear(D, D, bias=True).to(device)
    if basic_g:
        g = nn.Linear(D, 1, bias=True).to(device)
    else:
        g = nn.Linear(D, D, bias=True).to(device)

    g_x = g(x)
    f_x = f(x)

    w_ref = torch_scatter.scatter_softmax(g_x, jx, dim=dim)
    y_ref = torch_scatter.scatter_sum(f_x * w_ref, jx, dim=dim)

    w_om = om.torch_scatter_softmax(g_x.clone(), jx, dim=dim)
    y_om = om.torch_scatter_sum((f_x * w_om).clone(), jx, dim=dim)

    return jx, w_ref, w_om, y_ref, y_om


def assert_close(name, a, b, atol=1e-5, rtol=1e-5):
    d = (a - b).abs().max().item()
    ok = torch.allclose(a, b, atol=atol, rtol=rtol)
    print(f"{name}: max_abs_diff={d:.3e}  allclose={ok}")
    assert ok, f"{name} mismatch"


B, E, D = 2, 37, 64
for basic in (False, True):
    jx, wr, wo, yr, yo = run_softagg_pattern(B, E, D, basic_g=basic)
    print(f"--- SoftAgg{'Basic' if basic else ''} shapes: jx {tuple(jx.shape)}, w {tuple(wr.shape)}, y {tuple(yr.shape)} ---")
    assert_close("scatter_softmax", wr, wo)
    assert_close("scatter_sum", yr, yo)
print("OK: onnx_mods matches torch_scatter for SoftAgg pattern.")

--- SoftAgg shapes: jx (37,), w (2, 37, 64), y (2, 11, 64) ---
scatter_softmax: max_abs_diff=0.000e+00  allclose=True
scatter_sum: max_abs_diff=5.960e-08  allclose=True
--- SoftAggBasic shapes: jx (37,), w (2, 37, 1), y (2, 11, 64) ---
scatter_softmax: max_abs_diff=0.000e+00  allclose=True
scatter_sum: max_abs_diff=5.960e-08  allclose=True
OK: onnx_mods matches torch_scatter for SoftAgg pattern.


## 2. `scatter_max` (reduced shape vs `torch_scatter`)

`torch_scatter.scatter_max` returns a reduced tensor on `dim`; `onnx_mods.torch_scatter_max` does the same.

In [3]:
B, E, D = 2, 40, 32
dim = 1
ix = torch.randint(0, 9, (E,), device=device)
_, jx = torch.unique(ix, return_inverse=True)
src = torch.randn(B, E, D, device=device)

mr, _ = torch_scatter.scatter_max(src, jx, dim=dim)
mo, _ = om.torch_scatter_max(src.clone(), jx, dim=dim)
assert_close("scatter_max reduced", mr, mo)
print("OK.")

scatter_max reduced: max_abs_diff=0.000e+00  allclose=True
OK.


## 3. ONNX export smoke test

Tiny module that only uses `scatter_softmax` + `scatter_sum` with fixed `jx` (constant in the graph). Uses ops ONNX generally supports (`scatter_add`, `scatter_reduce`, `gather`, `div`).

In [4]:
try:
    import onnxruntime as ort
except ImportError:
    ort = None
    print("onnxruntime not installed; skipping ORT run.")


class ScatterSoftAggStub(nn.Module):
    """jx fixed length E; same math as SoftAgg core."""

    def __init__(self, d: int, e: int):
        super().__init__()
        self.f = nn.Linear(d, d)
        self.g = nn.Linear(d, d)
        ix = torch.randint(0, max(2, e // 4), (e,))
        _, jx = torch.unique(ix, return_inverse=True)
        self.register_buffer("jx", jx)

    def forward(self, x):
        jx = self.jx
        w = om.torch_scatter_softmax(self.g(x), jx, dim=1)
        y = om.torch_scatter_sum(self.f(x) * w, jx, dim=1)
        return y


d, e = 16, 24
m = ScatterSoftAggStub(d, e).to(device).eval()
x = torch.randn(1, e, d, device=device)
with torch.no_grad():
    y_pt = m(x)

buf = io.BytesIO()
torch.onnx.export(
    m,
    (x,),
    buf,
    input_names=["x"],
    output_names=["y"],
    opset_version=17,
    dynamo=False,
)
onnx_bytes = buf.getvalue()
print(f"ONNX export OK, size {len(onnx_bytes)} bytes, y shape {tuple(y_pt.shape)}")

if ort is not None:
    sess = ort.InferenceSession(onnx_bytes, providers=["CPUExecutionProvider"])
    y_ort = sess.run(None, {"x": x.cpu().numpy()})[0]
    err = abs(y_pt.cpu().numpy() - y_ort).max()
    print(f"ORT vs PyTorch max_abs_diff: {err:.3e}")
    assert err < 1e-3, "ORT numerical mismatch"

ONNX export OK, size 13034 bytes, y shape (1, 6, 16)
ORT vs PyTorch max_abs_diff: 2.384e-07


/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/dpvo/onnx_mods.py:151: TracerWarning: Converting a tensor to a Python number might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  dim_size = int(idx.max().item()) + 1
/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/dpvo/onnx_mods.py:93: TracerWarning: Converting a tensor to a Python number might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  return int(index.max().item()) + 1
2026-03-24 20:44:08.127047116 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. '/If_1_output_0' source:{1,6,16} target:{96}. Falling back to lenient merge.
